# Customer Churn Intelligence — Final Validation Model Comparison

## Objective

Consolidate and compare the **best validation results** from Steps 10–14. Select a leading model candidate (and optional backup) using **PR-AUC first**, then precision/recall tradeoff, then interpretability.

**Train + validation only — final test set is NOT used.**

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from xgboost import XGBClassifier

PROJECT_ROOT = Path("..").resolve()
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
FINAL_COMPARISON_PATH = REPORTS_DIR / "final_model_comparison.csv"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_split import load_split_from_manifest
from src.preprocessing import build_preprocessor

RANDOM_STATE = 42

## 1. Load Data & Define Best Model Configurations

In [ ]:
split = load_split_from_manifest()
X_train, X_val = split.X_train, split.X_val
y_train = (split.y_train == "Yes").astype(int)
y_val = (split.y_val == "Yes").astype(int)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

print(f"Train: {len(X_train):,} | Validation: {len(X_val):,} | Val churn rate: {y_val.mean():.2%}")

In [ ]:
def make_pipeline(model) -> ImbPipeline:
    return ImbPipeline([("preprocessor", build_preprocessor()), ("model", model)])


CANDIDATES = [
    {
        "Model": "DummyClassifier",
        "Imbalance_Strategy": "Most Frequent (baseline)",
        "estimator": DummyClassifier(strategy="most_frequent"),
    },
    {
        "Model": "LogisticRegression",
        "Imbalance_Strategy": "Original",
        "estimator": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    },
    {
        "Model": "LogisticRegression",
        "Imbalance_Strategy": "Class weighting",
        "estimator": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight="balanced"),
    },
    {
        "Model": "LogisticRegression",
        "Imbalance_Strategy": "RandomUnderSampler",
        "estimator": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
        "note": "Best Step-14 resampling variant for LR",
    },
    {
        "Model": "RandomForest",
        "Imbalance_Strategy": "Class weighting (tuned)",
        "estimator": RandomForestClassifier(
            n_estimators=200, max_depth=10, min_samples_leaf=5,
            class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1,
        ),
    },
    {
        "Model": "XGBoost",
        "Imbalance_Strategy": "Original",
        "estimator": XGBClassifier(
            n_estimators=200, max_depth=3, learning_rate=0.1, subsample=0.8,
            colsample_bytree=1.0, scale_pos_weight=1.0, random_state=RANDOM_STATE,
            eval_metric="logloss", n_jobs=-1,
        ),
    },
    {
        "Model": "XGBoost",
        "Imbalance_Strategy": "Class weighting (tuned)",
        "estimator": XGBClassifier(
            n_estimators=200, max_depth=3, learning_rate=0.1, subsample=0.8,
            colsample_bytree=1.0, scale_pos_weight=scale_pos_weight, random_state=RANDOM_STATE,
            eval_metric="logloss", n_jobs=-1,
        ),
    },
    {
        "Model": "XGBoost",
        "Imbalance_Strategy": "RandomUnderSampler",
        "estimator": XGBClassifier(
            n_estimators=200, max_depth=3, learning_rate=0.1, subsample=0.8,
            colsample_bytree=1.0, scale_pos_weight=1.0, random_state=RANDOM_STATE,
            eval_metric="logloss", n_jobs=-1,
        ),
        "note": "Best Step-14 resampling variant for XGBoost",
    },
]

len(CANDIDATES)

## 2. Train, Evaluate, and Build Comparison Table

In [ ]:
from imblearn.under_sampling import RandomUnderSampler

fitted = []
rows = []

for cfg in CANDIDATES:
    if cfg["Imbalance_Strategy"] == "RandomUnderSampler":
        pipe = ImbPipeline([
            ("preprocessor", build_preprocessor()),
            ("undersample", RandomUnderSampler(random_state=RANDOM_STATE)),
            ("model", cfg["estimator"]),
        ])
    else:
        pipe = make_pipeline(cfg["estimator"])

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_val)
    y_proba = pipe.predict_proba(X_val)[:, 1]

    label = f"{cfg['Model']} | {cfg['Imbalance_Strategy']}"
    fitted.append({"label": label, "pipe": pipe, "y_pred": y_pred, "y_proba": y_proba, **cfg})

    rows.append({
        "Model": cfg["Model"],
        "Imbalance_Strategy": cfg["Imbalance_Strategy"],
        "Accuracy": round(accuracy_score(y_val, y_pred), 4),
        "Precision": round(precision_score(y_val, y_pred, zero_division=0), 4),
        "Recall": round(recall_score(y_val, y_pred), 4),
        "F1": round(f1_score(y_val, y_pred), 4),
        "ROC_AUC": round(roc_auc_score(y_val, y_proba), 4),
        "PR_AUC": round(average_precision_score(y_val, y_proba), 4),
    })

comparison_df = pd.DataFrame(rows).sort_values("PR_AUC", ascending=False).reset_index(drop=True)
comparison_df["Rank"] = comparison_df.index + 1
comparison_df.to_csv(FINAL_COMPARISON_PATH, index=False)

print(f"Saved: {FINAL_COMPARISON_PATH}")
comparison_df

## 3. ROC & Precision-Recall Curves (Leading Models)

In [ ]:
TOP_N = 5
top_labels = comparison_df.head(TOP_N).apply(
    lambda r: f"{r['Model']} | {r['Imbalance_Strategy']}", axis=1
).tolist()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for item in fitted:
    if item["label"] not in top_labels:
        continue
    fpr, tpr, _ = roc_curve(y_val, item["y_proba"])
    prec, rec, _ = precision_recall_curve(y_val, item["y_proba"])
    pr_auc = average_precision_score(y_val, item["y_proba"])
    roc_auc = roc_auc_score(y_val, item["y_proba"])

    axes[0].plot(fpr, tpr, label=f"{item['label']} (AUC={roc_auc:.3f})")
    axes[1].plot(rec, prec, label=f"{item['label']} (AP={pr_auc:.3f})")

axes[0].plot([0, 1], [0, 1], "k--", alpha=0.4)
axes[1].axhline(y_val.mean(), color="k", linestyle="--", alpha=0.4, label="Prevalence baseline")
axes[0].set_title("ROC Curves — Top Models")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[1].set_title("Precision-Recall Curves — Top Models")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
for ax in axes:
    ax.legend(fontsize=7, loc="lower right")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "08_model_comparison_roc_pr_curves.png", dpi=120)
plt.show()

## 4. Confusion Matrix Comparison (Top 3 by PR-AUC)

In [ ]:
top3_labels = comparison_df.head(3).apply(
    lambda r: f"{r['Model']} | {r['Imbalance_Strategy']}", axis=1
).tolist()

cms = []
for item in fitted:
    if item["label"] in top3_labels:
        cm = confusion_matrix(y_val, item["y_pred"])
        cms.append({
            "Model": item["label"],
            "TN": cm[0, 0], "FP": cm[0, 1],
            "FN": cm[1, 0], "TP": cm[1, 1],
        })

pd.DataFrame(cms)

## 5. Model Selection

### Ranking criteria (in order)
1. **PR-AUC** — most informative for imbalanced churn (~26.5% positive class)
2. **Useful precision/recall tradeoff** at default threshold 0.5
3. **Simplicity / interpretability** when performance is close

### Selected candidates

**Leading model: XGBoost + Class weighting (tuned)**
- Highest validation **PR-AUC** and strong **Recall** for retention use cases
- Captures nonlinear interactions; `scale_pos_weight` handles imbalance without synthetic data
- Not chosen on accuracy alone (0.762 vs 0.806 for default LR)

**Backup model: Logistic Regression + Class weighting**
- Near-top **PR-AUC** (within ~0.008 of XGBoost) with **interpretable coefficients**
- Preferred when stakeholders need explainability and performance is close

### Why others were not selected

| Model | Reason not selected |
|-------|---------------------|
| **DummyClassifier** | Zero recall — no churn detection value |
| **LogisticRegression (Original)** | Higher accuracy/precision but **misses ~41% of churners** (Recall 0.59) |
| **RandomForest (tuned)** | Best **F1** (0.654) but lower **PR-AUC** than XGBoost; less ranking power for prioritization |
| **Resampling variants (RUS/SMOTENC)** | Did not beat class weighting on PR-AUC; SMOTENC hurt ranking metrics |
| **XGBoost (Original)** | Lower recall and PR-AUC than class-weighted version |

**Test set remains untouched.** Threshold optimization and calibration are deferred to later steps.

In [ ]:
leading = comparison_df.iloc[0]
backup = comparison_df[
    (comparison_df["Model"] == "LogisticRegression")
    & (comparison_df["Imbalance_Strategy"] == "Class weighting")
].iloc[0]

selection = pd.DataFrame([
    {"Role": "Leading candidate", **leading.to_dict()},
    {"Role": "Backup candidate", **backup.to_dict()},
])
selection